In [28]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [ ]:
catalog = dbutils.widgets.get("catalog")
checkpoints_dir = dbutils.widgets.get("checkpoints_dir")

In [27]:
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, TimestampType, StringType, DateType
from pyspark.sql import functions as F
import sys
import os

home = os.path.abspath(os.path.join(os.getcwd(), "..","..",".."))

sys.path.append(home)

from src.utils.json_parser import parse_json

In [ ]:
df = spark.readStream.table(f"{catalog}.brz.employees_raw")

schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("employee_id", LongType(), True),
    StructField("before", StructType([
        StructField("employee_name", StringType(), True),
        StructField("role", StringType(), True),
        StructField("department", StringType(), True),
        StructField("region_id", IntegerType(), True),
        StructField("joining_date", DateType(), True),
        StructField("salary", LongType(), True)
    ]), True),
    StructField("after", StructType([
        StructField("employee_name", StringType(), True),
        StructField("role", StringType(), True),
        StructField("department", StringType(), True),
        StructField("region_id", IntegerType(), True),
        StructField("joining_date", DateType(), True),
        StructField("salary", LongType(), True)
    ]), True),
    StructField("operation", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

parsed_df = parse_json(spark, df, "value", schema)

parsed_df = (parsed_df.select("parsed.event_id", "parsed.employee_id",
                                F.col("parsed.before.employee_name").alias("before_employee_name"),
                                F.col("parsed.before.role").alias("before_role"),
                                F.col("parsed.before.department").alias("before_department"),
                                F.col("parsed.before.region_id").alias("before_region_id"),
                                F.col("parsed.before.joining_date").alias("before_joining_date"),
                                F.col("parsed.before.salary").alias("before_salary"),
                                F.col("parsed.after.employee_name").alias("after_employee_name"),
                                F.col("parsed.after.role").alias("after_role"),
                                F.col("parsed.after.department").alias("after_department"),
                                F.col("parsed.after.region_id").alias("after_region_id"),
                                F.col("parsed.after.joining_date").alias("after_joining_date"),
                                F.col("parsed.after.salary").alias("after_salary"),
                                "parsed.operation", "parsed.event_time",
                                F.current_timestamp().alias("processed_time"))
                                )

query = (parsed_df.writeStream
        .format("delta")
        .option("checkpointLocation", f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints_dir}/brz_checkpoints/employee_cdc_checkpoint")
        .trigger(availableNow = True)
        .outputMode("append")
        .table(f"{catalog}.brz.employee_cdc"))

query.awaitTermination(100)